# Data Warehousing Solutions

In [ ]:
import math
import numpy as np
import pandas as pd

import psycopg2


In [ ]:
#
# function to run a select query and return rows in a pandas dataframe
# pandas puts all numeric values from postgres to float
# if it will fit in an integer, change it to integer
#

def my_select_query_pandas(query, rollback_before_flag, rollback_after_flag):
    "function to run a select query and return rows in a pandas dataframe"
    
    if rollback_before_flag:
        connection.rollback()
    
    df = pd.read_sql_query(query, connection)
    
    if rollback_after_flag:
        connection.rollback()
    
    # fix the float columns that really should be integers
    
    for column in df:
    
        if df[column].dtype == "float64":

            fraction_flag = False

            for value in df[column].values:
                
                if not np.isnan(value):
                    if value - math.floor(value) != 0:
                        fraction_flag = True

            if not fraction_flag:
                df[column] = df[column].astype('Int64')
    
    return(df)
    

In [ ]:
connection = psycopg2.connect(
    user = "postgres",
    password = "ucb",
    host = "postgres",
    port = "5432",
    database = "postgres"
)

In [ ]:
cursor = connection.cursor()

##  You try it - using the star schema, for each receipt, find the subtotal, tax, and total amounts

In [ ]:
rollback_before_flag = True
rollback_after_flag = True

query = """

select r.receipt,
       sum(l.line_item_sub_total) as sub_total,
       sum(l.line_item_tax) as tax,
       sum(l.line_item_total) as total
from line_item_facts as l
     join receipt_dimension as r
         on l.receipt_key = r.receipt_key
group by r.receipt
order by r.receipt

"""

my_select_query_pandas(query, rollback_before_flag, rollback_after_flag)

##  You try it - find another extra rows problem and another missing rows problem when joining the orders star schema and the fulfillment star schema

##  order 4 has the extra rows problem, as it was fulfilled in fullfillments 14 and 15

In [ ]:
rollback_before_flag = True
rollback_after_flag = True

query = """

select o.order_id,
       o.order_date,
       o.sub_total,
       o.tax,
       o.total,
       f.fulfillment_id,
       f.fulfillment_date
from orders as o
     join fulfillment as f
         on o.order_id = f.order_id
where o.order_id = 4
order by 1

"""

my_select_query_pandas(query, rollback_before_flag, rollback_after_flag)

##  order 4 will be double counted

In [ ]:
rollback_before_flag = True
rollback_after_flag = True

query = """

select o.order_id,
       o.order_date,
       sum(o.sub_total) as sub_total,
       sum(o.tax) as tax,
       sum(o.total) as total
from orders as o
     join fulfillment as f
         on o.order_id = f.order_id
where o.order_id = 4
group by o.order_id, o.order_date
order by 1

"""

my_select_query_pandas(query, rollback_before_flag, rollback_after_flag)

##  fulfillment 15 fulfilled orders 5 and 6, with 5 as primary;  if we query for the fulfillment for order 6, it will be missing

In [ ]:
rollback_before_flag = True
rollback_after_flag = True

query = """

select o.order_id,
       o.order_date,
       o.sub_total,
       o.tax,
       o.total,
       f.fulfillment_id,
       f.fulfillment_date
from orders as o
     join fulfillment as f
         on o.order_id = f.order_id
where o.order_id = 6
order by 1

"""

my_select_query_pandas(query, rollback_before_flag, rollback_after_flag)